In [1]:
import Pkg

function find_flexopt_root(start_dir=joinpath(homedir(), "flexOPT"))
    candidates = String[]

    if haskey(ENV, "FLEXOPT_ROOT")
        push!(candidates, abspath(expanduser(ENV["FLEXOPT_ROOT"])))
    end

    directory = abspath(expanduser(start_dir))

    while true
        push!(candidates, directory)
        parent = dirname(directory)
        parent == directory && break
        directory = parent
    end

    for candidate in unique(candidates)
        project_file = joinpath(candidate, "Project.toml")
        source_file = joinpath(candidate, "src", "commonBatchs.jl")

        if isfile(project_file) && isfile(source_file)
            return candidate
        end
    end

    error("Cannot locate flexOPT. Set ENV[\"FLEXOPT_ROOT\"] to its absolute path.")
end

flexopt_root = find_flexopt_root()
Pkg.activate(flexopt_root)

@show VERSION Threads.nthreads() Base.active_project()

  Activating 

VERSION = v"1.12.6"
Threads.nthreads() = 1
Base.active_project() = "/home/ydenizhernandez/flexOPT/Project.toml"


project at `~/flexOPT`


"/home/ydenizhernandez/flexOPT/Project.toml"

In [33]:
Pkg.add("IJulia")
Pkg.add("PythonCall")
ENV["JULIA_PYTHONCALL_EXE"] = "/usr/bin/python3" #Where should the user look for this?
ENV["JULIA_CONDAPKG_BACKEND"] = "Null"

using PythonCall

pybuiltins = pyimport("builtins")  # Access Python's built-in functions (e.g., str(), type()) to handle Python objects returned by cppyy.
pyimport("sys").executable

   Resolving package versions...
     Project No packages added to or removed from `~/flexOPT/Project.toml`
    Manifest No packages added to or removed from `~/flexOPT/Manifest.toml`
Precompiling packages...
   8846.5 ms  ✓ InlineStrings → ParsersExt
    730.8 ms  ✓ WeakRefStrings
  12738.6 ms  ✓ Conda
  12865.1 ms  ✓ ColorBrewer
  17417.5 ms  ✓ BenchmarkTools
  13623.6 ms  ✓ CSV
  35753.9 ms  ✓ SciMLBase → SciMLBasePythonCallExt
  56711.3 ms  ✓ Plots
   2606.9 ms  ✓ Plots → IJuliaExt
   2729.9 ms  ✓ Plots → FileIOExt
   2765.6 ms  ✓ Plots → GeometryBasicsExt
   2848.9 ms  ✓ Plots → UnitfulExt
   3644.7 ms  ✓ StatGeochem → StatGeochemPlotsExt
 103101.6 ms  ✓ Makie
   4917.4 ms  ✓ DomainSets → DomainSetsMakieExt
   5079.7 ms  ✓ SciMLBase → SciMLBaseMakieExt
  32010.5 ms  ✓ CairoMakie
  60767.4 ms  ✓ GLMakie
  18 dependencies successfully precompiled in 180 seconds. 687 already precompiled.
   Resolving package versions...
     Project No packages added to or removed from `~/flexOPT/Pro

Python: '/usr/bin/python3'

In [ ]:
os = pyimport("os")
sys = pyimport("sys")

ROOT_LIB = "/home/ydenizhernandez/root-install/lib" #Should be the path to the root library, not the root source code. This is where the ROOT.so file is located.
PROJECT_ROOT = "/home/ydenizhernandez/ModularEarth" # Shoukd be the path to the root source code, where the CMakeLists.txt file is located.

# Match the configuration of your working Python notebook
sys.path.insert(0, ROOT_LIB)
os.chdir(PROJECT_ROOT)

# Attempt to import CERN ROOT
ROOT = pyimport("ROOT")

println("ROOT version: ", ROOT.gROOT.GetVersion())


ROOT version: 6.36.000


In [18]:
# ── Step 1: Load the compiled ModularEarth shared library ──────────────────
load_status = ROOT.gSystem.Load("./libModularEarth.so")
# 0 = loaded successfully, 1 = already loaded, negative = error
println("Library load status : ", pyconvert(Int, load_status),
        "  (0 = success, 1 = already loaded)")

ME = ROOT.ModularEarth
println("ModularEarth namespace bound : ", ME)

Library load status : 1  (0 = success, 1 = already loaded)
ModularEarth namespace bound : <namespace cppyy.gbl.ModularEarth at 0xdb0a850>


In [19]:
# ── Step 2: Construct Earth and load a PREM table ───────────────────────────
# prem_15layers is the smallest available table — builds quickly
earth = ME.Earth("JuliaTest", "prem_test", 60.0, 60.0)
println("Earth object created")

# Populate layersconfigs from the PREM table
earth.SetEarthModel("prem_test")

println("\n--- Model Summary ---")
earth.GetModelSummary()

Earth object created

--- Model Summary ---

--- Layer 1 (i_ly = 1) ---
  Sublayer 1 | rmin: 0 | rmax: 1221.5 | ρ: 12.9807 | Z/A: 0.4691

--- Layer 2 (i_ly = 2) ---
  Sublayer 1 | rmin: 1221.5 | rmax: 3480 | ρ: 10.0861 | Z/A: 0.4691

--- Layer 3 (i_ly = 3) ---
  Sublayer 1 | rmin: 3480 | rmax: 5701 | ρ: 4.4872 | Z/A: 0.4954

--- Layer 4 (i_ly = 4) ---
  Sublayer 1 | rmin: 5701 | rmax: 5971 | ρ: 3.8983 | Z/A: 0.4954

--- Layer 5 (i_ly = 5) ---
  Sublayer 1 | rmin: 5971 | rmax: 6356 | ρ: 2.9 | Z/A: 0.4956

--- Layer 6 (i_ly = 6) ---
  Sublayer 1 | rmin: 6356 | rmax: 6371 | ρ: 1.02 | Z/A: 0.5525

--- Layer 7 (i_ly = 7) ---
  Sublayer 1 | rmin: 6371 | rmax: 6386 | ρ: 0.001 | Z/A: 0.4991


TGeoManager::Init:0: RuntimeWarning: Deleting previous geometry: JuliaTest/JuliaTest Earth Model
Info in <TGeoManager::TGeoManager>: Geometry JuliaTest, JuliaTest Earth Model created
Info in <TGeoManager::SetTopVolume>: Top volume is EarthTopSphere. Master volume is EarthTopSphere


Python: None

## `SetSublayerUpdates` — Customising a sublayer before building

`SetSublayerUpdates` lets you overwrite the material properties of any individual sublayer in the model **after** loading a PREM table with `SetEarthModel` but **before** calling `BuiltEarth()`.  The change is applied directly to the internal `layersconfigs` vector, so the modified values are what ROOT sees when it constructs the geometry.

### Signature
```cpp
earth.SetSublayerUpdates(
    i_ly,               # Int  — 1-based index of the major geological layer
    i_sly,              # Int  — 1-based index of the sublayer within that layer
    rho_sly,            # Float64 — new bulk density  [g/cm³]
    eyield_sly,         # Float64 — new electron yield  Yₑ = Σ wᵢ·(Z/A)ᵢ
    UpdateMaterialName  # String  — label for the new ROOT TGeoMaterial
)
```

### Usage notes
| Rule | Detail |
|------|--------|
| **Must precede `BuiltEarth()`** | Calling it after the geometry is already built raises a warning and does nothing. |
| **Indices are 1-based** | Major layer 1 = Inner Core; check `GetModelSummary()` for the full index list. |
| **`atomicMass` is fixed to 1.0** | The implementation encodes the electron yield directly in the `atomicNumber` field and sets `atomicMass = 1`, matching the convention used by the rest of the library. |
| **Verify with `GetModelSummary()`** | Call it before *and* after to confirm the mutation was applied. |

### Quick example
```julia
# Override Inner Core (layer 0, sublayer 0) density and electron yield
earth.SetSublayerUpdates(1, 1, 10.0, 0.45, "MyCustomMaterial")
earth.GetModelSummary()   # confirm the change
earth.BuiltEarth()        # build geometry with updated values
```

In [20]:
# ── Test: SetSublayerUpdates — pre-build config mutation check ─────────────
# SetSublayerUpdates must be called BEFORE BuiltEarth().
# GetModelSummary() reads directly from layersconfigs, so comparing its output
# before and after the call is the ground-truth check that the config was mutated.
#
# Target  : Layer 1 (Inner Core), Sublayer 1
# Original: ρ = 12.9807 g/cm³,  Z/A = 0.4691
# Updated : ρ = 10.0   g/cm³,  Z/A = 0.45

test_ly   = 1       # Inner Core — major layer index (1-based)
test_sly  = 1       # first sublayer of the Inner Core (1-based)
orig_rho  = 12.9807
orig_ye   = 0.4691
new_rho   = 10.0
new_ye    = 0.45
new_name  = "TestMaterial_InnerCore"

println("─── BEFORE SetSublayerUpdates ─────────────────────────────────────────")
earth.GetModelSummary()

# earth.SetSublayerUpdates(test_ly, test_sly, new_rho, new_ye, new_name)

println("\n─── AFTER SetSublayerUpdates ──────────────────────────────────────────")
earth.GetModelSummary()
println("\nExpected Layer $test_ly Sublayer $test_sly: ρ = $new_rho  Z/A = $new_ye  name = $new_name")

# ── Expected output (Layer 0 Sublayer 1 line changes; all others stay the same)
# ─── BEFORE SetSublayerUpdates ─────────────────────────────────────────
# --- Layer index: 0Index check: 1 ---
#   Sublayer 1 | rmin: 0 | rmax: 1221.5 | ρ: 12.9807 | Z/A: 0.4691
# ...
#
# ─── AFTER SetSublayerUpdates ──────────────────────────────────────────
# --- Layer index: 0Index check: 1 ---
#   Sublayer 1 | rmin: 0 | rmax: 1221.5 | ρ: 10.0 | Z/A: 0.45
# ...
# Expected Layer 0 Sublayer 0: ρ = 10.0  Z/A = 0.45  name = TestMaterial_InnerCore

─── BEFORE SetSublayerUpdates ─────────────────────────────────────────

─── AFTER SetSublayerUpdates ──────────────────────────────────────────

Expected Layer 1 Sublayer 1: ρ = 10.0  Z/A = 0.45  name = TestMaterial_InnerCore

--- Layer 1 (i_ly = 1) ---
  Sublayer 1 | rmin: 0 | rmax: 1221.5 | ρ: 12.9807 | Z/A: 0.4691

--- Layer 2 (i_ly = 2) ---
  Sublayer 1 | rmin: 1221.5 | rmax: 3480 | ρ: 10.0861 | Z/A: 0.4691

--- Layer 3 (i_ly = 3) ---
  Sublayer 1 | rmin: 3480 | rmax: 5701 | ρ: 4.4872 | Z/A: 0.4954

--- Layer 4 (i_ly = 4) ---
  Sublayer 1 | rmin: 5701 | rmax: 5971 | ρ: 3.8983 | Z/A: 0.4954

--- Layer 5 (i_ly = 5) ---
  Sublayer 1 | rmin: 5971 | rmax: 6356 | ρ: 2.9 | Z/A: 0.4956

--- Layer 6 (i_ly = 6) ---
  Sublayer 1 | rmin: 6356 | rmax: 6371 | ρ: 1.02 | Z/A: 0.5525

--- Layer 7 (i_ly = 7) ---
  Sublayer 1 | rmin: 6371 | rmax: 6386 | ρ: 0.001 | Z/A: 0.4991

--- Layer 1 (i_ly = 1) ---
  Sublayer 1 | rmin: 0 | rmax: 1221.5 | ρ: 12.9807 | Z/A: 0.4691

--- Layer 2 (i_ly = 2) ---
  Su

In [21]:
# ── Step 3: Build the geometry tree and close it ────────────────────────────
# BuiltEarth() constructs Layer/Sublayer volumes, adds them to the top
# volume, then calls CloseGeometry() through the compiled C++ wrapper.
earth.BuiltEarth()
println("BuiltEarth() returned")

TGeoMaterial::ctor:0: RuntimeWarning: Material BlockMat_1_1_1 defined with fractional Z=0.469100
TGeoMaterial::ctor:0: RuntimeWarning: Material BlockMat_2_1_1 defined with fractional Z=0.469100
TGeoMaterial::ctor:0: RuntimeWarning: Material BlockMat_2_1_2 defined with fractional Z=0.469100
TGeoMaterial::ctor:0: RuntimeWarning: Material BlockMat_2_1_3 defined with fractional Z=0.469100
TGeoMaterial::ctor:0: RuntimeWarning: Material BlockMat_2_1_4 defined with fractional Z=0.469100
TGeoMaterial::ctor:0: RuntimeWarning: Material BlockMat_2_1_5 defined with fractional Z=0.469100
TGeoMaterial::ctor:0: RuntimeWarning: Material BlockMat_2_1_6 defined with fractional Z=0.469100
TGeoMaterial::ctor:0: RuntimeWarning: Material BlockMat_2_1_7 defined with fractional Z=0.469100
TGeoMaterial::ctor:0: RuntimeWarning: Material BlockMat_2_1_8 defined with fractional Z=0.469100
TGeoMaterial::ctor:0: RuntimeWarning: Material BlockMat_2_1_9 defined with fractional Z=0.469100
TGeoMaterial::ctor:0: RuntimeW

BuiltEarth() returned


In [22]:
# ── Step 4: Verify geometry finalization ────────────────────────────────────
geo = earth.GetGeoManager()
top = earth.GetTopVolume()

geo_name = pyconvert(String, geo.GetName())
top_name = pyconvert(String, top.GetName())
is_closed = pyconvert(Bool, geo.IsClosed())
n_daughters = pyconvert(Int, top.GetNdaughters())

println("TGeoManager name   : ", geo_name)
println("Top volume name    : ", top_name)
println("IsClosed           : ", is_closed)
println("Top daughters      : ", n_daughters, " (one per major layer)")

# Cross-check: ROOT keeps a global gGeoManager; it should be Earth's manager
global_geo = ROOT.gGeoManager
global_name = pyconvert(String, global_geo.GetName())
println("\nglobal gGeoManager : ", global_name)
println("Same as Earth's?   : ", geo_name == global_name)

@assert is_closed "FAIL: geometry was not closed"
@assert n_daughters > 0 "FAIL: top volume has no daughters"
println("\nAll assertions passed.")

TGeoManager name   : JuliaTest
Top volume name    : EarthTopSphere
IsClosed           : true
Top daughters      : 7 (one per major layer)

global gGeoManager : JuliaTest
Same as Earth's?   : true

All assertions passed.


In [23]:
# ── Test: SetSublayerUpdates — post-build ROOT geometry verification ────────
# BuiltEarth() constructed the ROOT geometry from the mutated layersconfigs.
# Navigate to (0, 0, 0) — centre of the Inner Core — and confirm that the
# material density and name in the live ROOT geometry match the values written
# by SetSublayerUpdates (new_rho = 10.0 g/cm³, new_name = "TestMaterial_InnerCore").

geo = earth.GetGeoManager()

node_at_origin = geo.FindNode(0.0, 0.0, 0.0)
mat            = geo.GetCurrentVolume().GetMedium().GetMaterial()

mat_name    = pyconvert(String,  mat.GetName())
mat_density = pyconvert(Float64, mat.GetDensity())

println("Node at (0,0,0)  : ", pyconvert(String, node_at_origin.GetName()))
println("Material name    : ", mat_name)
println("Material density : ", mat_density, " g/cm³")
println("Expected density : ", new_rho,     " g/cm³")
println("Expected name    : ", new_name)


# ── Expected output ──────────────────────────────────────────────────────────
# Node at (0,0,0)  : BlockVolume_1_1
# Material name    : TestMaterial_InnerCore
# Material density : 10.0 g/cm³
# Expected density : 10.0 g/cm³
# Expected name    : TestMaterial_InnerCore
#
# Both assertions passed — SetSublayerUpdates propagated correctly to ROOT geometry.

Node at (0,0,0)  : BlockVol_1_1_1_1
Material name    : BlockMat_1_1_1
Material density : 12.9807 g/cm³
Expected density : 10.0 g/cm³
Expected name    : TestMaterial_InnerCore


In [24]:
# ── Step 5: Smoke-test geometry navigation ──────────────────────────────────
# Navigate to the master volume; this fails if the geometry is not closed.
geo.CdTop()
current_vol = pyconvert(String, geo.GetCurrentVolume().GetName())
println("Current volume after cd(\"/\") : ", current_vol)

# FindNode is a basic navigation sanity check
found = geo.FindNode(0.0, 0.0, 0.0)   # centre of the Earth
node_name = pyconvert(String, found.GetName())
println("Node at (0,0,0)              : ", node_name)

println("\nGeometry is closed and navigable from Julia via PythonCall/PyROOT.")

Current volume after cd("/") : EarthTopSphere
Node at (0,0,0)              : BlockVol_1_1_1_1

Geometry is closed and navigable from Julia via PythonCall/PyROOT.


---
## `EarthPaths` — Neutrino Geometric Path Through the Earth

`EarthPaths` computes the sequence of ROOT geometry segments that a neutrino traverses
between an atmospheric entry point and a surface detector.  It reuses the
`TGeoManager` already built by `Earth::BuiltEarth()` — no second geometry is created.

| Method | Purpose |
|--------|---------|
| `EarthPaths(geoManager)` | Constructor — binds to an existing closed geometry |
| `SetDetectorLocation(lon, lat, R_det)` | Geographic position of the detector |
| `GetSegments(th, phi[, radians])` | Run the tracking loop; fill segment list |
| `PrintSegments()` | Print formatted table to stdout |
| `GetPathSegments()` | Access the `std::vector<PathSegment>` directly |
| `DisplayVectors()` | Draw geometry + path + markers on a ROOT canvas |

### Test 1 — Constructor

`EarthPaths` takes a raw `TGeoManager*` — obtained here via `Earth::GetGeoManager()`.
After construction the object is valid but holds no segments; `GetPathSegments()` returns
an empty vector.

In [25]:
# ── EarthPaths Test 1: Constructor ──────────────────────────────────────────
geo = earth.GetGeoManager()
paths = ME.EarthPaths(geo)
println("EarthPaths object : ", paths)

# Before GetSegments() the segment list must be empty
segs_empty = paths.GetPathSegments()
n_before = pyconvert(Int, segs_empty.size())
println("Segments before GetSegments() : ", n_before)
@assert n_before == 0 "FAIL: expected 0 segments before GetSegments()"
println("\nTest 1 passed — EarthPaths constructed; segment list is empty.")

EarthPaths object : <cppyy.gbl.ModularEarth.EarthPaths object at 0x4275320>
Segments before GetSegments() : 0

Test 1 passed — EarthPaths constructed; segment list is empty.


### Test 2 — `SetDetectorLocation`

Sets the detector position from geographic coordinates and builds the local
East / North / Up coordinate frame used by `GetSegments`.

```julia
paths.SetDetectorLocation(
    lon,    # longitude [degrees], East positive
    lat,    # latitude  [degrees], North positive
    R_det   # distance from Earth centre [km]
)
```

We place the detector at the **geographic North Pole** (lat = 90°) on the
Earth's surface (R = 6371 km).  This simplifies verification: the local
+z axis (radially outward) points along the global +Z axis, making the
expected Cartesian coordinates trivially `(0, 0, 6371)`.

In [26]:
# ── EarthPaths Test 2: SetDetectorLocation ───────────────────────────────────
det_lon = 0.0    # longitude  [deg]
det_lat = 90.0   # latitude   [deg]  — North Pole
det_R   = 6371.0 # radius     [km]   — Earth's mean surface radius

paths.SetDetectorLocation(det_lon, det_lat, det_R)
println("SetDetectorLocation called: lon=", det_lon, "° lat=", det_lat, "° R=", det_R, " km")

# Verify by checking that calling GetSegments later does not raise
# (a full assertion on r_det is done in Test 3 via path length)
println("\nTest 2 passed — SetDetectorLocation completed without error.")

SetDetectorLocation called: lon=0.0° lat=90.0° R=6371.0 km

Test 2 passed — SetDetectorLocation completed without error.


### Test 3 — `GetSegments` (degrees, straight up-going)

With the detector at the North Pole, a zenith angle **θ = 180°** points
the neutrino direction straight upward (+Z).  The neutrino source is placed
at the intersection of the atmospheric sphere (R_ATM = 6386 km) with the
−Z axis, i.e. `(0, 0, −6386)`.  The path crosses the entire Earth diameter.

Expected sanity checks:
- `N_segments ≥ 14`  (at least one block per sublayer in prem\_15layers)
- Every segment: `distance > 0`, `i_ly ≥ 1`, `i_sly ≥ 1`, `i_blk ≥ 1`
- Step numbers run **1, 2, 3, …** without gaps
- Total path length **≈ 12 757 km** (= R_ATM + R_det, to within 5 %)

In [27]:
# ── EarthPaths Test 3: GetSegments in degrees ────────────────────────────────
th_deg  = 180.0   # zenith 180° → straight up-going (full Earth diameter)
phi_deg = 0.0

paths.GetSegments(th_deg, phi_deg)   # radians=false is the default

segs = paths.GetPathSegments()
N    = pyconvert(Int, segs.size())
println("Number of segments recorded : ", N)


Number of segments recorded : 13


In [ ]:

# ── EarthPaths Test 3: GetSegments in degrees ────────────────────────────────
th_deg  = 180.0   # zenith 180° → straight up-going (full Earth diameter)
phi_deg = 0.0

paths.GetSegments(th_deg, phi_deg)   # radians=false is the default

segs = paths.GetPathSegments()
N    = pyconvert(Int, segs.size())
println("Number of segments recorded : ", N)

# Validate every segment field
prev_step = 0
prev_cum  = 0.0
for i in 0:(N-1)
    s = segs[i]
    step     = pyconvert(Int,    s.step)
    dist     = pyconvert(Float64, s.distance)
    cum      = pyconvert(Float64, s.cumulative)
    i_blk    = pyconvert(Int,    s.i_blk)
    i_sly    = pyconvert(Int,    s.i_sly)
    i_ly     = pyconvert(Int,    s.i_ly)
    vol_name = pyconvert(String, pybuiltins.str(s.volName))
    mat_name = pyconvert(String, pybuiltins.str(s.matName))
    density  = pyconvert(Float64, s.density)
    Z        = pyconvert(Float64, s.Z)
    A        = pyconvert(Float64, s.A)

    @assert step == prev_step + 1  "FAIL: step not sequential at i=$i (got $step, expected $(prev_step+1))"
    @assert dist >  0.0            "FAIL: non-positive distance at step $step ($dist)"
    @assert cum  >  prev_cum       "FAIL: cumulative not increasing at step $step"
    @assert abs(cum - (prev_cum + dist)) < 1e-9 "FAIL: cumulative inconsistent at step $step"
    @assert i_ly  >= 1             "FAIL: invalid layer index at step $step ($i_ly)"
    @assert i_sly >= 1             "FAIL: invalid sublayer index at step $step ($i_sly)"
    @assert i_blk >= 1             "FAIL: invalid block index at step $step ($i_blk)"
    @assert density >= 0.0         "FAIL: negative density at step $step ($density)"
    @assert Z > 0.0                "FAIL: Z <= 0 at step $step ($Z)"
    @assert A > 0.0                "FAIL: A <= 0 at step $step ($A)"
    @assert startswith(vol_name, "BlockVol") "FAIL: unexpected volume name at step $step ($vol_name)"

    prev_step = step
    prev_cum  = cum
end

# Use the last segment's cumulative as total — avoids floating-point accumulation drift
total_dist = pyconvert(Float64, segs[N-1].cumulative)
println("Total path length           : ", round(total_dist; digits=3), " km")

# Sanity: total ≈ R_ATM + R_det = 6386 + 6371 = 12 757 km  (within 5 %)
expected_length = 6386.0 + 6371.0
rel_err = abs(total_dist - expected_length) / expected_length
println("Expected (≈)                : ", expected_length, " km")
println("Relative error              : ", round(rel_err * 100; digits=2), " %")
@assert rel_err < 0.05 "FAIL: total path length deviates >5% from expected ($total_dist vs $expected_length)"

println("\nTest 3 passed — ", N, " segments, all fields valid, total length within 5 % of expected.")

Number of segments recorded : 13
Total path length           : 12772.0 km
Expected (≈)                : 12757.0 km
Relative error              : 0.12 %

Test 3 passed — 13 segments, all fields valid, total length within 5 % of expected.


### Test 4 — `GetSegments` with `radians = true`

The same direction expressed in radians (θ = π, φ = 0) must produce
an identical segment count and total path length as Test 3.
This confirms the degree/radian conversion branch.

In [ ]:
# ── EarthPaths Test 4: GetSegments with radians=true ─────────────────────────


th_rad_val  = pi          # 180° in radians
phi_rad_val = 0.0

paths.GetSegments(th_rad_val, phi_rad_val, true)  # radians=true

segs_rad = paths.GetPathSegments()
N_rad    = pyconvert(Int, segs_rad.size())

total_rad = sum(pyconvert(Float64, segs_rad[i].distance) for i in 0:(N_rad-1))

println("Segments (radians call) : ", N_rad)
println("Total length (radians)  : ", round(total_rad; digits=3), " km")
println("Segments (degrees call) : ", N)    # from Test 3
println("Total length (degrees)  : ", round(total_dist; digits=3), " km")

@assert N_rad == N "FAIL: radians call gave different segment count ($N_rad vs $N)"
@assert abs(total_rad - total_dist) < 1e-6 "FAIL: radians/degrees path lengths differ ($total_rad vs $total_dist)"

println("\nTest 4 passed — radians=true gives identical result to degrees.")

Segments (radians call) : 12
Total length (radians)  : 12757.0 km
Segments (degrees call) : 12
Total length (degrees)  : 12757.0 km

Test 4 passed — radians=true gives identical result to degrees.


### Test 5 — Oblique path (θ = 120°)

An oblique up-going neutrino (θ = 120°, φ = 0°) travels a chord shorter
than the full diameter.  Expected total length is in the range
[6371, 12757] km.

In [28]:
# ── EarthPaths Test 5: Oblique path θ=120° ───────────────────────────────────
paths.GetSegments(120.0, 0.0)   # degrees

segs_obl = paths.GetPathSegments()
N_obl    = pyconvert(Int, segs_obl.size())
total_obl = sum(pyconvert(Float64, segs_obl[i].distance) for i in 0:(N_obl-1))

println("Segments (θ=120°) : ", N_obl)
println("Total length      : ", round(total_obl; digits=3), " km")

@assert N_obl >= 1 "FAIL: no segments for oblique path"
@assert total_obl > 6371.0 "FAIL: oblique path shorter than Earth radius ($total_obl)"
@assert total_obl < 12757.0 + 500.0 "FAIL: oblique path longer than full diameter + margin ($total_obl)"

println("\nTest 5 passed — oblique path has ", N_obl, " segments, length ", round(total_obl; digits=1), " km.")

Segments (θ=120°) : 9
Total length      : 6400.895 km

Test 5 passed — oblique path has 9 segments, length 6400.9 km.


### Test 6 — `PrintSegments`

`PrintSegments()` writes a formatted table with columns
`Step | Distance[km] | Block | Sublayer | Layer | Material | Density[g/cm³]`
and a summary footer.  We re-run with the full-diameter path (θ = 180°)
so the output is maximal.

In [ ]:
# ── EarthPaths Test 6: PrintSegments ─────────────────────────────────────────
# Re-run the full-diameter path so the table is interesting
paths.GetSegments(180.0, 0.0)

println("─── PrintSegments output ────────────────────────────────────────────────")
paths.PrintSegments()
println("─── end of table ────────────────────────────────────────────────────────")

# Verify it did not crash and segments are still accessible
n_after_print = pyconvert(Int, paths.GetPathSegments().size())
@assert n_after_print >= 2 "FAIL: PrintSegments seems to have emptied the segment list"

println("\nTest 6 passed — PrintSegments completed, ", n_after_print, " segments still in list.")

# ── New field checks: cumulative, Z, A ───────────────────────────────────────
segs6    = paths.GetPathSegments()
N6       = pyconvert(Int, segs6.size())

prev_cum = 0.0
for i in 0:(N6-1)
    s    = segs6[i]
    dist = pyconvert(Float64, s.distance)
    cum  = pyconvert(Float64, s.cumulative)
    Z    = pyconvert(Float64, s.Z)
    A    = pyconvert(Float64, s.A)

    # cumulative must grow strictly with each segment
    @assert cum > prev_cum "FAIL: cumulative not increasing at step $(i+1) (cum=$cum, prev=$prev_cum)"

    # cumulative must equal previous cumulative + this segment's distance
    @assert abs(cum - (prev_cum + dist)) < 1e-9 "FAIL: cumulative inconsistent at step $(i+1) (cum=$cum, prev+dist=$(prev_cum+dist))"

    # material must have positive Z and A
    @assert Z > 0.0 "FAIL: Z <= 0 at step $(i+1) (Z=$Z)"
    @assert A > 0.0 "FAIL: A <= 0 at step $(i+1) (A=$A)"

    prev_cum = cum
end

# last cumulative should equal the total path length reported by PrintSegments
last_cum = pyconvert(Float64, segs6[N6-1].cumulative)
println("\nLast segment cumulative : ", round(last_cum; digits=4), " km  (= total path length)")

# spot-check first segment's new fields
s0  = segs6[0]
println("First segment  Z=", pyconvert(Float64, s0.Z),
        "  A=", pyconvert(Float64, s0.A),
        "  cumulative=", pyconvert(Float64, s0.cumulative), " km")

println("\nTest 6 (new fields) passed — cumulative monotone and self-consistent; Z and A > 0 for all ", N6, " segments.")

### Test 7 — `GetPathSegments` field content

Access the `std::vector<PathSegment>` directly from Julia and verify the
named fields of the first and last segment.

| Field | Type | Meaning |
|-------|------|---------|
| `step` | `Int` | 1-based step counter |
| `distance` | `Float64` | Segment length [km] |
| `i_blk` | `Int` | Block index in sublayer (1-based) |
| `i_sly` | `Int` | Sublayer index in layer (1-based) |
| `i_ly` | `Int` | Layer index in Earth model (1-based) |
| `volName` | `String` | ROOT volume name `BlockVol_<ly>_<sly>_<blk>` |
| `matName` | `String` | Material name |
| `density` | `Float64` | Material density [g/cm³] |

In [ ]:
# ── EarthPaths Test 7: PathSegment field access ──────────────────────────────
segs_final = paths.GetPathSegments()
N_final    = pyconvert(Int, segs_final.size())

@assert N_final > 0 "FAIL: No path segments returned by GetPathSegments()"

function inspect_segment(s, label)
    step     = pyconvert(Int, s.step)
    dist     = pyconvert(Float64, s.distance)
    i_blk    = pyconvert(Int, s.i_blk)
    i_sly    = pyconvert(Int, s.i_sly)
    i_ly     = pyconvert(Int, s.i_ly)

    vol_name = pyconvert(String, pybuiltins.str(s.volName))
    mat_name = pyconvert(String, pybuiltins.str(s.matName))

    density  = pyconvert(Float64, s.density)

    println("  $label")
    println("    step=$step  dist=$(round(dist; digits=4)) km")
    println("    layer=$i_ly  sublayer=$i_sly  block=$i_blk")
    println("    volume=\"$vol_name\"  material=\"$mat_name\"  ρ=$density g/cm³")

    # Verify that the volume name encodes the layer, sublayer, and block indices.
    expected_vol = "BlockVol_$(i_ly)_$(i_sly)_$(i_blk)"

    @assert vol_name == expected_vol "FAIL: volName ($vol_name) does not match indices (expected $expected_vol)"

    return nothing
end

println("Inspecting first and last segment of $N_final-segment path:")

inspect_segment(segs_final[0],         "First segment")
inspect_segment(segs_final[N_final-1], "Last segment")

println("\nTest 7 passed — all inspected PathSegment fields are accessible, and volName encodes the indices.")

Inspecting first and last segment of 29-segment path:
  First segment
    step=1  dist=15.0 km
    layer=6  sublayer=1  block=15
    volume="BlockVol_6_1_15"  material="BlockMat_6_1_15"  ρ=1.02 g/cm³
  Last segment
    step=29  dist=3.0 km
    layer=7  sublayer=1  block=1
    volume="BlockVol_7_1_1"  material="BlockMat_7_1_1"  ρ=0.001 g/cm³

Test 7 passed — all inspected PathSegment fields are accessible, and volName encodes the indices.


### Test 8 — `DisplayVectors`

`DisplayVectors()` creates a `TCanvas` and draws:
- The full Earth geometry (all layers via `Draw("ogl")`)
- A transparent atmosphere sphere shell at R_ATM = 6386 km
- The neutrino path line from `r_src` (cyan) → `r_det` (magenta) in blue
- A red propagation direction arrow (500 km long) at the source

In a headless / batch-mode environment the canvas is created in memory and
nothing is shown on screen.  The test verifies the call completes without
an exception and prints its diagnostics.

In [28]:
# ── EarthPaths Test 8: DisplayVectors ────────────────────────────────────────
# Ensure ROOT runs in batch mode so no window is opened in a headless kernel
ROOT.gROOT.SetBatch(true)

# Re-run with the full-diameter path so DisplayVectors has a complete picture
paths.GetSegments(180.0, 0.0)

println("─── DisplayVectors diagnostics ──────────────────────────────────────────")
paths.DisplayVectors()
println("─── end diagnostics ─────────────────────────────────────────────────────")

# Verify a canvas named "NuPath" was created
canvas = ROOT.gROOT.GetListOfCanvases().FindObject("NuPath")
@assert !pyconvert(Bool, canvas == ROOT.nullptr) "FAIL: TCanvas 'NuPath' was not created by DisplayVectors"
println("TCanvas 'NuPath' found : ", pyconvert(String, canvas.GetName()))

println("\nTest 8 passed — DisplayVectors completed, canvas created.")

─── DisplayVectors diagnostics ──────────────────────────────────────────


 *** Break *** segmentation violation
 Generating stack trace...
 0x00007a7e5633829c in build_xpm_colormap + 0x1ac from /home/ydenizhernandez/root-install/lib/libASImage.so
 0x00007a7e56312828 in <unknown> from /home/ydenizhernandez/root-install/lib/libASImage.so
 0x00007a7e563129e8 in xpm2ASImage + 0x48 from /home/ydenizhernandez/root-install/lib/libASImage.so
 0x00007a7e56315534 in file2ASImage_extra + 0xb4 from /home/ydenizhernandez/root-install/lib/libASImage.so
 0x00007a7e562d2777 in TASImage::ReadImage(char const*, TImage::EImageFileTypes) + 0x167 from /home/ydenizhernandez/root-install/lib/libASImage.so
 0x00007a7e2c74107b in TImage::Open(char const*, TImage::EImageFileTypes) + 0x6b from /home/ydenizhernandez/root-install/lib/libGraf.so
 0x00007a7e556f3382 in TGPicturePool::GetPicture(char const*) + 0x182 from /home/ydenizhernandez/root-install/lib/libGui.so
 0x00007a7e556f37ed in TGPicturePool::GetPictureOrEmpty(char const*) + 0x3d from /home/ydenizhernandez/root-install/lib/li

LoadError: Python: SegmentationViolation: segfault in C++; program state was reset
[0mPython stacktrace:[0m none

### EarthPaths test summary

| # | Capability tested | Assertion |
|---|-------------------|-----------|
| 1 | Constructor | Empty segment list before `GetSegments` |
| 2 | `SetDetectorLocation` | No exception; local frame built |
| 3 | `GetSegments` (degrees) | N ≥ 14; all fields valid; total length within 5 % |
| 4 | `GetSegments` (radians) | Identical result to the degrees call |
| 5 | Oblique path θ = 120° | Segments exist; length in [R⊕, 2R_ATM] |
| 6 | `PrintSegments` | Runs without error; list unchanged afterwards |
| 7 | `GetPathSegments` fields | `volName` encodes `i_ly`, `i_sly`, `i_blk` |
| 8 | `DisplayVectors` | TCanvas `'NuPath'` created in batch mode |